# Đánh giá checkpoint F0–F4 bằng VisDrone DET metric

Notebook này **không train lại**. Add Input gồm processed VisDrone, raw `VisDrone2019-DET-val`, và Kaggle Output chứa `best.pth` của các experiment cần đánh giá. Bật GPU + Internet rồi Run All.


In [ ]:
from pathlib import Path

EXPERIMENTS = ["F0", "F1", "F2", "F3", "F4"]
CHECKPOINT_OVERRIDES = {}  # ví dụ: {"F4": "/kaggle/input/.../f4/best.pth"}
PROCESSED_DATA_ROOT = None  # để None cho notebook tự tìm
RAW_VAL_ROOT = None         # thư mục có images/ và annotations/ gốc
OUTPUT_ROOT = Path("/kaggle/working/visdrone_official_eval")
REPO_URL = "https://github.com/Nhattk19/Object_dectection.git"
REPO_BRANCH = "cnn-faster-rcnn-pipeline"


In [ ]:
import subprocess, sys

matches = list(Path("/kaggle/working").glob("**/scripts/evaluate_visdrone_checkpoint.py"))
matches += list(Path("/kaggle/input").glob("**/scripts/evaluate_visdrone_checkpoint.py"))
if matches:
    PROJECT_ROOT = matches[0].parents[1].resolve()
else:
    PROJECT_ROOT = Path("/kaggle/working/Object_dectection")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch",
                           REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)])
import torch
assert torch.cuda.is_available(), "Vào Settings bật GPU Accelerator"
print({"project": str(PROJECT_ROOT), "gpu": torch.cuda.get_device_name(0)})


In [ ]:
def valid_processed(path):
    path = Path(path)
    return (path / "annotations/instances_val.json").is_file() and (path / "images/val").is_dir()

def valid_raw_val(path):
    path = Path(path)
    return (path / "images").is_dir() and len(list((path / "annotations").glob("*.txt"))) >= 500

if PROCESSED_DATA_ROOT:
    DATA_ROOT = Path(PROCESSED_DATA_ROOT)
else:
    DATA_ROOT = next((p.parent.parent for p in Path("/kaggle/input").glob("**/annotations/instances_val.json") if valid_processed(p.parent.parent)), None)
if RAW_VAL_ROOT:
    RAW_ROOT = Path(RAW_VAL_ROOT)
else:
    RAW_ROOT = next((p.parent for p in Path("/kaggle/input").glob("**/annotations") if valid_raw_val(p.parent)), None)
if DATA_ROOT is None or not valid_processed(DATA_ROOT):
    raise FileNotFoundError("Không tìm thấy processed VisDrone validation")
if RAW_ROOT is None or not valid_raw_val(RAW_ROOT):
    raise FileNotFoundError("Không tìm thấy raw VisDrone2019-DET-val (cần annotations gốc)")
print({"processed": str(DATA_ROOT), "raw_val": str(RAW_ROOT)})


In [ ]:
def find_checkpoint(experiment):
    if experiment in CHECKPOINT_OVERRIDES:
        path = Path(CHECKPOINT_OVERRIDES[experiment])
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates = [p for p in Path("/kaggle/input").glob("**/best.pth")
                  if p.parent.name.lower() == experiment.lower() and "smoke" not in str(p).lower()]
    candidates.sort(key=lambda p: ("faster_rcnn_runs" not in str(p), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy best.pth của {experiment}; Add Output hoặc đặt CHECKPOINT_OVERRIDES")
    if len(candidates) > 1:
        print(f"{experiment}: có nhiều checkpoint, chọn {candidates[0]}")
    return candidates[0]

CHECKPOINTS = {experiment: find_checkpoint(experiment) for experiment in EXPERIMENTS}
CHECKPOINTS


In [ ]:
for experiment, checkpoint in CHECKPOINTS.items():
    command = [sys.executable, "-u", str(PROJECT_ROOT / "scripts/evaluate_visdrone_checkpoint.py"),
               "--checkpoint", str(checkpoint), "--data-root", str(DATA_ROOT),
               "--raw-val-root", str(RAW_ROOT), "--output-dir", str(OUTPUT_ROOT),
               "--workers", "2"]
    print("Evaluating", experiment, checkpoint)
    subprocess.check_call(command, cwd=PROJECT_ROOT)


In [ ]:
import json, pandas as pd
from IPython.display import display
reports = [json.loads((OUTPUT_ROOT / experiment.lower() / "visdrone_metrics.json").read_text()) for experiment in EXPERIMENTS]
columns = ["experiment", "checkpoint_epoch", "visdrone_ap", "visdrone_ap50", "visdrone_ap75", "visdrone_ar1", "visdrone_ar10", "visdrone_ar100", "visdrone_ar500"]
display(pd.DataFrame(reports)[columns].sort_values("visdrone_ap", ascending=False))
print("Save Version để giữ kết quả:", OUTPUT_ROOT)
